## Named Entity Recognition

In [1]:
import pickle
import time

import numpy as np
import pandas as pd
import requests
import spacy
from sklearn.neighbors import BallTree
from spacy.lang.en.stop_words import STOP_WORDS

from helper import read_config

In [ ]:
# Apply the NER model to our dataset (list of abstracts)
data = pd.read_csv('../get_data/alldata.csv')
abstracts = data["Abstract"]

# Define NLP model (only the named-entity recognizer is needed)
nlp = spacy.load('en_core_web_trf', disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])

# Process every abstract
processed_abstracts = [nlp(abstract) for abstract in abstracts]

In [ ]:
# Save the processed docs in chunks (pickling all docs at once runs out of memory)
processed_data_path = '../NER/alldata_processed.pkl'
chunk_size = 1000  # Adjust based on your memory constraints

with open(processed_data_path, 'wb') as file:
    for i in range(0, len(processed_abstracts), chunk_size):
        pickle.dump(processed_abstracts[i:i + chunk_size], file)

In [ ]:
# Get all locations (GPE and LOC) from the processed documents

# Dictionary to store location entities for each abstract
locations = {}

# Extract location entities from saved docs
for idx, doc in enumerate(processed_abstracts):
    location_set = set()
    for ent in doc.ents:
        if ent.label_ == 'GPE' or ent.label_ == 'LOC':
            words = ent.text.split()
            filtered_location = " ".join(word for word in words if word.lower() not in STOP_WORDS)
            if filtered_location:
                location_set.add(filtered_location)

    locations[idx] = list(location_set)

# Combine unique locations from all abstracts into a single set
all_locations = set()
for loc_list in locations.values():
    all_locations.update(loc_list)
print("Number of unique locations:", len(all_locations))

# Save the locations for each document (one row per paper, same order as alldata.csv)
data_loc = pd.DataFrame.from_dict(locations, orient='index')
data_loc.to_csv('../NER/data_loc.csv', index=False)

# Save all_locations as a CSV file using pandas
all_locations_df = pd.DataFrame({'Location': list(all_locations)})
all_locations_df.to_csv('../NER/all_locations.csv', index=False)

## Get coordinates for each location using the GeoNames API

In [ ]:
# GeoNames username is read from configurations.ini (not version controlled; see configurations.ini.example)
username = read_config()['geonames']['username']


def search_location(location_name, fuzzy=False):
    params = {"name": location_name, "maxRows": 50, "username": username}
    if fuzzy:
        params["fuzzy"] = 0.8
    return requests.get("http://api.geonames.org/searchJSON", params=params).json()


def geocode(location_name):
    """Exact search, then fuzzy search, then fuzzy search on each word. Returns None if nothing is found."""
    data = search_location(location_name)
    if data.get("geonames"):
        match, kind = data["geonames"][0], "exact"
    else:
        data = search_location(location_name, fuzzy=True)
        if data.get("geonames"):
            match, kind = data["geonames"][0], "fuzzy"
        else:
            for word in location_name.split():
                data = search_location(word, fuzzy=True)
                if data.get("geonames"):
                    match = data["geonames"][0]
                    kind = match.get("name", "Unknown")
                    break
            else:
                return None
    return {
        "location": kind,
        "feature_class": match.get("fcl", "Unknown"),
        "latitude": match.get("lat", "Unknown"),
        "longitude": match.get("lng", "Unknown"),
    }


def geocode_all(location_names, delay):
    """Geocode each name, waiting `delay` seconds between locations (API rate limits). Returns (found, not_found)."""
    found, not_found = {}, []
    for location_name in location_names:
        try:
            result = geocode(location_name)
            if result is None:
                print(f"Location '{location_name}' not found in GeoNames database")
        except Exception as e:
            print(f"An error occurred for location '{location_name}': {e}")
            result = None
        if result is None:
            not_found.append(location_name)
        else:
            found[location_name] = result
        time.sleep(delay)
    return found, not_found


def save_locations(location_data, error_locations):
    location_df = pd.DataFrame.from_dict(location_data, orient='index')
    location_df.index.name = 'Location'
    location_df.to_csv('../NER/fuzzy_location_data.csv')
    pd.DataFrame({'Error Locations': error_locations}).to_csv('../NER/fuzzy_error_locations.csv', index=False)


# Geocode every unique location
all_loc = pd.read_csv('../NER/all_locations.csv')['Location']
location_data, error_locations = geocode_all(all_loc, delay=2)
save_locations(location_data, error_locations)

In [ ]:
# Retry the locations that failed (e.g. because of API rate limits) with a longer delay
error_locations = list(dict.fromkeys(pd.read_csv('../NER/fuzzy_error_locations.csv')['Error Locations']))
location_data = pd.read_csv('../NER/fuzzy_location_data.csv', float_precision='round_trip').set_index('Location').to_dict(orient='index')

found, error_locations = geocode_all(error_locations, delay=5)
location_data.update(found)
save_locations(location_data, error_locations)
print(f"{len(found)} locations found on retry, {len(error_locations)} still not found")

In [2]:
# Manual coordinates for locations that GeoNames could not find or placed wrongly
error_dict = {"Arafura-Timor Seas":[-9.5, 134.5], "Punnakayal":[8.632222, 78.111950],
              "Aguincheira":[40.835621, -8.380748], "Rhine-Ruhr":[51.451770, 7.018720],
              "Qinghai-Tibet Plateau":[35.751230, 94.988449], "Fukutoku-Okanoba": [34.004161, 134.563027],
              "Odaw":[5.548512, -0.224942], "Chellanam":[9.807211, 76.277420], "South Malang":[-7.979470, 112.634222],
              "Cabo-Delgado":[-12.750000, 39.500000], "Rhine-Meuse":[51.730431, 4.715880],
              "Ho-Chi-Minh city": [10.668271, 106.482262], "Solimões":[-2.601662, -55.143238],
              "Santos Bay":[-23.959779, -46.344984], "Perú":[-11.185900, -76.841576], "North Maluku Province":[1.348434, 127.668530],
              "Patok-Fushëkuqe-Ishëm":[41.628820, 19.619820], "High-Arctic":[79.138640, -3.238970], "Sanbanze":[35.672459, 139.967468],
              "Apulo-Lucanian":[40.371390, 16.811220], "Lohabarrack":[11.603486, 92.609825], "Alto-Tiete":[-23.060740, -47.723861],
              "Kinmen-Xiamen":[24.479720, 118.081900], "Kodiyaghat":[-2.780561, -78.854599]}

location_df = pd.read_csv('../NER/fuzzy_location_data.csv', float_precision='round_trip').set_index('Location')
for name, (lat, lon) in error_dict.items():
    location_df.loc[name, ['location', 'feature_class', 'latitude', 'longitude']] = ['manual', 'manual', lat, lon]
    # Clear the basin so that it is recomputed for the new coordinates
    if 'Closest Ocean Basin' in location_df.columns:
        location_df.loc[name, ['Closest Ocean Basin', 'Distance to Nearest Ocean']] = np.nan
location_df.reset_index().to_csv('../NER/fuzzy_location_data.csv', index=False)

# Remove the manually located places from the error list
error_locations = pd.read_csv('../NER/fuzzy_error_locations.csv')['Error Locations']
error_locations = error_locations[~error_locations.isin(list(error_dict))].drop_duplicates()
error_locations.to_csv('../NER/fuzzy_error_locations.csv', index=False)
print(f"{len(error_dict)} locations set manually; {len(error_locations)} locations remain unresolved")

24 locations set manually; 127 locations remain unresolved


## Attribute each location to an ocean basin

In [3]:
# Assign each location to the closest cell of the 1-degree ocean-basin grid (great-circle distance)
location_df = pd.read_csv('../NER/fuzzy_location_data.csv', float_precision='round_trip')
range_area = pd.read_csv('../NER/range_area.csv')

# Remove grid cells with invalid coordinates
range_area = range_area[range_area['Latitude'].between(-90, 90) & range_area['Longitude'].between(-180, 180)]

# Only locations without a basin are (re)computed: new locations and manually corrected ones.
# Locations exactly equidistant to two grid cells are tie-broken differently across scikit-learn
# versions, so existing assignments are kept to make the classification reproducible.
if 'Closest Ocean Basin' not in location_df.columns:
    location_df['Closest Ocean Basin'] = np.nan
    location_df['Distance to Nearest Ocean'] = np.nan
todo = location_df['Closest Ocean Basin'].isna()

if todo.any():
    tree = BallTree(np.radians(range_area[['Latitude', 'Longitude']].values), metric='haversine')
    coords = np.radians(location_df.loc[todo, ['latitude', 'longitude']].values.astype(float))
    distances, indices = tree.query(coords, k=1)
    location_df.loc[todo, 'Closest Ocean Basin'] = range_area['range_area'].values[indices[:, 0]]
    location_df.loc[todo, 'Distance to Nearest Ocean'] = distances[:, 0] * 6371  # radians -> km
location_df['Closest Ocean Basin'] = location_df['Closest Ocean Basin'].astype(int)

location_df.to_csv('../NER/fuzzy_location_data.csv', index=False)
print(f"Assigned {todo.sum()} locations to a basin")
location_df

Assigned 24 locations to a basin


,Location,location,feature_class,latitude,longitude,Closest Ocean Basin,Distance to Nearest Ocean
0,Liao River,exact,H,40.684070,122.148960,9,36.020823
1,Corsica,exact,P,43.425270,-98.407300,3,11.181412
2,Mazandaran,exact,L,36.000000,52.000000,15,71.423607
3,Tambakrejo,exact,P,-7.270600,111.621600,11,28.817852
4,Samut Prakan Province,exact,A,13.604420,100.705310,15,25.047528
...,...,...,...,...,...,...,...
6484,Apulo-Lucanian,manual,manual,40.371390,16.811220,22,29.971653
6485,Lohabarrack,manual,manual,11.603486,92.609825,14,16.600176
6486,Alto-Tiete,manual,manual,-23.060740,-47.723861,7,53.930643
6487,Kinmen-Xiamen,manual,manual,24.479720,118.081900,9,42.368094


In [4]:
# Number of locations by distance to the nearest ocean-basin grid cell
bins = [0, 50, 100, 500, 1000, np.inf]
labels = ['< 50 km', '50-100 km', '100-500 km', '500-1000 km', '>= 1000 km']
pd.cut(location_df['Distance to Nearest Ocean'], bins=bins, labels=labels, right=False).value_counts().sort_index()

Distance to Nearest Ocean
< 50 km        4259
50-100 km      2155
100-500 km       75
500-1000 km       0
>= 1000 km        0
Name: count, dtype: int64

## Classify each paper by ocean basin

In [5]:
data_loc = pd.read_csv('../NER/data_loc.csv')
location_df = pd.read_csv('../NER/fuzzy_location_data.csv', float_precision='round_trip')
data_loc.head()

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Pacific,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Replace each location by the code of its closest basin. Locations that could not be geocoded
# become NaN, so that they do not take part in the vote below.
location_dict = location_df.set_index('Location')['Closest Ocean Basin'].to_dict()
labels_df = data_loc.map(lambda val: location_dict.get(val, np.nan) if pd.notna(val) else val)
labels_df

,0,1,2,3,4,5,6,7,8,9,...,12,13,14,15,16,17,18,19,20,21
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,10.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12393,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12394,7.0,19.0,7.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12395,8.0,8.0,10.0,9.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
12396,8.0,3.0,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
# Classify each paper by its most frequent basin (None if none of its locations was found)
classes = [row.dropna().value_counts().idxmax() if row.notna().any() else None
           for _, row in labels_df.iterrows()]
classes_df = pd.DataFrame({'Most Recurrent Value': pd.array(classes, dtype='Int64')})
classes_df

,Most Recurrent Value
0,<NA>
1,<NA>
2,<NA>
3,<NA>
4,10
...,...
12393,<NA>
12394,7
12395,8
12396,8


In [8]:
## Fix some issues with the basins dataset
range_area = pd.read_csv('../NER/range_area.csv')

# Convert the 'range_area' column to float
range_area['range_area'] = range_area['range_area'].astype(float)

# Filter instances with latitude outside the valid range
invalid_latitudes = range_area[(range_area['Latitude'] < -90) | (range_area['Latitude'] > 90)]

# Filter instances with longitude outside the valid range
invalid_longitudes = range_area[(range_area['Longitude'] < -180) | (range_area['Longitude'] > 180)]

# Get the indices of instances with invalid latitude or longitude
invalid_indices = invalid_latitudes.index.union(invalid_longitudes.index)

# Remove instances with invalid latitude or longitude from range_area
range_area = range_area.drop(invalid_indices)

## Remove differentiation between coastal and oceanic basins (i.e. coastal north atlantic and north atlantic)
merge_classes = [[2,3,1], [4,5], [6,7],[8,9], [10,11], [12,13], [14,15], [16,17], [18,19]]

# Create the mapping dictionary for merging
merge_mapping = {}
for merged_class in merge_classes:
    merged_label = merged_class[0]
    for class_label in merged_class:
        merge_mapping[class_label] = merged_label

# Merge the classes based on the mapping
classes_df['Merged Class'] = classes_df['Most Recurrent Value'].replace(merge_mapping)

# Calculate the basin counts
basin_counts = classes_df['Merged Class'].value_counts()

# Merge the classes based on the mapping in range_area DataFrame
range_area['Merged Class'] = range_area['range_area'].replace(merge_mapping)

# Map the basin counts to the 'range_area' column
range_area['basin_count'] = range_area['Merged Class'].map(basin_counts)

# Include ocean label besides code
range_basin_list = pd.read_csv('../NER/range_basin_list.csv', names=['range_area', 'basin'])
range_basin_list['range_area'] = range_basin_list['range_area'].astype(float).round(1)
range_area = range_area.merge(range_basin_list, on='range_area', how='left')
range_area.drop(['range_area','Merged Class'], axis=1, inplace=True)

# Create a dictionary to map range_area to basin
label_mapping = dict(zip(range_basin_list['range_area'], range_basin_list['basin']))

# Map the values in 'merged_classes' to labels
classes_df['Basin'] = classes_df['Merged Class'].map(label_mapping)

range_area.to_csv('../NER/maps/range_area_counts.csv', index=False)
classes_df.to_csv('../NER/merged_fuzzy_classes_labels.csv', index=False)

In [9]:
classes_df['Basin'].dropna().unique()

array(['Equatorial Pac', 'North Atlantic', 'North Pacific', 'Arctic',
       'Antarctic', 'Mediteranean', 'North Indian', 'Equatorial Indian',
       'Baltic Sea', 'South Pacific', 'Equatorial Atlant',
       'South Atlantic', 'South Indian', 'Persian Gulf', 'Red Sea',
       'Black Sea', 'Sulu Sea'], dtype=object)